# R4 — Entrenamiento y selección sobre validation

## Qué corrige

En la v1, `train_and_evaluate` calculaba el umbral óptimo por F1 **sobre el
mismo conjunto donde reportaba las métricas**, y los 182 modelos se comparaban
entre sí sobre ese mismo conjunto. Dos sesgos apilados: ajuste de umbral y
selección de modelo, ambos sobre el conjunto de reporte.

Aquí **todo se decide sobre validation**. El test no se abre en este notebook.

In [1]:
# Celda de arranque idéntica en todos los notebooks. Deja el kernel de SageMaker
# en un estado conocido: mismo directorio de trabajo, mismas semillas, mismas
# versiones. Si algo de esto cambia entre corridas, los resultados no son
# comparables aunque los notebooks sean los mismos.
import sys, os

AQUI = os.getcwd()                    # los notebooks y vishing_common.py conviven
if AQUI not in sys.path:
    sys.path.insert(0, AQUI)

import numpy as np
import pandas as pd
import vishing_common as vc

vc.set_all_seeds()                    # random, numpy y torch (+ cuDNN determinista)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

print("dataset:", vc.RAW_FILENAME, "(esquema", vc.DATASET_VERSION + ")")
print("bucket :", vc.BUCKET, "| prefijo:", vc.PREFIX)
print("seed   :", vc.SEED, "| split:", vc.SPLIT_MODE, "| política:", vc.FEATURE_POLICY)
print("xgboost se ejecutará en:", vc.xgb_device())
vc.check_versions()

dataset: biocatch_sinthetic_data_v3.csv (esquema v3)
bucket : poc-vishing | prefijo: v2
seed   : 42 | split: grouped | política: audited
xgboost se ejecutará en: cuda
  versiones OK: numpy 2.0.2, pandas 2.2.3, scipy 1.14.1, sklearn 1.5.2, imblearn 0.12.4, xgboost 2.1.4


,paquete,instalada,esperado,ok
0,numpy,2.0.2,">=1.26,<2.1",True
1,pandas,2.2.3,">=2.1,<2.3",True
2,scipy,1.14.1,">=1.11,<1.15",True
3,sklearn,1.5.2,">=1.4,<1.6",True
4,imblearn,0.12.4,">=0.12,<0.13",True
5,xgboost,2.1.4,">=2.0,<2.2",True


In [2]:
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

cfg = vc.read_json(vc.P.feature_contract)
contract = cfg["contratos"][cfg["activo"]]

val = vc.read_parquet(vc.P.val)
Xv = val[contract["features"]].fillna(0).values
yv = val[vc.TARGET].values
print("validation:", Xv.shape, "| vishing: %.4f" % yv.mean())

validation: (19673, 44) | vishing: 0.0508


In [3]:
VARIANTES_XGB = {
    "xgb_base":        dict(max_depth=6, learning_rate=0.1, n_estimators=100),
    "xgb_deep":        dict(max_depth=9, learning_rate=0.05, n_estimators=300,
                            min_child_weight=3),
    "xgb_shallow":     dict(max_depth=3, learning_rate=0.1, n_estimators=500),
    "xgb_regularized": dict(max_depth=6, learning_rate=0.1, n_estimators=200,
                            reg_alpha=1.0, reg_lambda=5.0, min_child_weight=10,
                            gamma=0.3),
    "xgb_balanced":    dict(max_depth=6, learning_rate=0.1, n_estimators=200,
                            scale_pos_weight=None),
    "xgb_conservative": dict(max_depth=6, learning_rate=0.1, n_estimators=200,
                             subsample=0.7, colsample_bytree=0.7, gamma=0.5),
    "xgb_slow_learner": dict(learning_rate=0.01, n_estimators=500, max_depth=6,
                             subsample=0.8),
}
# device se resuelve en tiempo de ejecución: 'cuda' si la instancia tiene GPU,
# 'cpu' si no. XGBoost >= 2.0 falla con device='cuda' en una instancia sin GPU,
# y este notebook se ejecuta a veces en CPU para una prueba rápida.
BASE_XGB = vc.xgb_base_params()
print("XGBoost:", BASE_XGB)

OTRAS_FAMILIAS = {
    "logistic_regression": lambda: LogisticRegression(max_iter=1000,
                                                      random_state=vc.SEED),
    "random_forest": lambda: RandomForestClassifier(n_estimators=150, max_depth=10,
                                                    random_state=vc.SEED, n_jobs=-1),
    "mlp": lambda: MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300,
                                 random_state=vc.SEED),
}
NECESITAN_ESCALA = {"logistic_regression", "mlp"}

XGBoost: {'tree_method': 'hist', 'device': 'cuda', 'eval_metric': 'logloss', 'random_state': 42, 'n_jobs': -1}


In [4]:
def entrenar_y_validar(nombre, hacer_modelo, Xt, yt, escalar, meta):
    """Entrena sobre train, calcula umbral y métricas SOBRE VALIDATION."""
    sc = None
    Xt_, Xv_ = Xt, Xv
    if escalar:
        sc = StandardScaler().fit(Xt)
        Xt_, Xv_ = sc.transform(Xt), sc.transform(Xv)

    modelo = hacer_modelo()
    if isinstance(modelo, XGBClassifier) and modelo.get_params().get("scale_pos_weight") is None:
        n_neg, n_pos = int((yt == 0).sum()), int((yt == 1).sum())
        modelo.set_params(scale_pos_weight=round(n_neg / max(n_pos, 1), 2))

    modelo.fit(Xt_, yt)
    pv = modelo.predict_proba(Xv_)[:, 1]

    thr = vc.f1_optimal_threshold(yv, pv)            # <-- umbral SOBRE VALIDATION
    rep = vc.full_report(yv, pv, thr, n_boot=0)      # IC solo en la evaluación final

    envoltorio = vc.VishingModelWrapper(
        model=modelo, contract=contract, threshold=thr, scaler=sc,
        metadata={**meta, "variant": nombre, "threshold_fitted_on": "validation"})
    vc.write_pickle(envoltorio, vc.P.model(meta["data_type"], nombre,
                                           meta["technique"], meta["ratio"]))
    return {**meta, "variante": nombre, "umbral": round(thr, 4),
            "pr_auc": round(rep["pr_auc"], 4), "roc_auc": round(rep["roc_auc"], 4),
            "recall": round(rep["recall"], 4), "precision": round(rep["precision"], 4),
            "f1": round(rep["f1"], 4),
            "recall_p90": round(rep["recall_at_precision_90"]["recall"], 4)}

## Barrido completo

In [ ]:
resultados = []

for data_type in ["original", "augmented"]:
    rutas = vc.list_parquet(vc.P.balanced_dir(data_type))
    print("\n##### %s: %d datasets #####" % (data_type, len(rutas)))

    for uri in rutas:
        partes = uri.rstrip("/").split("/")
        tecnica, ratio = partes[-2], partes[-1].replace(".parquet", "")
        d = vc.read_parquet(uri)
        Xt = d[contract["features"]].fillna(0).values
        yt = d[vc.TARGET].values
        meta = {"data_type": data_type, "technique": tecnica, "ratio": ratio}
        print("  %-22s %-4s (%d filas)" % (tecnica, ratio, len(d)))

        for v, params in VARIANTES_XGB.items():
            resultados.append(entrenar_y_validar(
                v, lambda p=params: XGBClassifier(**{**BASE_XGB, **p}),
                Xt, yt, False, meta))
        for v, hacer in OTRAS_FAMILIAS.items():
            resultados.append(entrenar_y_validar(
                v, hacer, Xt, yt, v in NECESITAN_ESCALA, meta))

# Orden determinista: sin criterios de desempate, dos configuraciones con el
# mismo PR-AUC redondeado dejarían al ganador a merced del orden de iteración,
# y R5 y R6 heredarían esa arbitrariedad.
lb = (pd.DataFrame(resultados)
      .sort_values(["pr_auc", "recall_p90", "roc_auc", "f1",
                    "data_type", "technique", "ratio", "variante"],
                   ascending=[False, False, False, False, True, True, True, True])
      .reset_index(drop=True))
print("\ntotal de modelos entrenados:", len(lb))
display(lb.head(25))
vc.write_csv(lb, vc.P.val_leaderboard)


##### original: 13 datasets #####
  borderline_smote       10   (63472 filas)


/home/ec2-user/SageMaker/vishing-remediation/vishing-venv/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [16:31:14] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


  escrito s3://poc-vishing/v2/05_models/original/xgb_base/borderline_smote/10.pkl
  escrito s3://poc-vishing/v2/05_models/original/xgb_deep/borderline_smote/10.pkl
  escrito s3://poc-vishing/v2/05_models/original/xgb_shallow/borderline_smote/10.pkl
  escrito s3://poc-vishing/v2/05_models/original/xgb_regularized/borderline_smote/10.pkl
  escrito s3://poc-vishing/v2/05_models/original/xgb_balanced/borderline_smote/10.pkl
  escrito s3://poc-vishing/v2/05_models/original/xgb_conservative/borderline_smote/10.pkl
  escrito s3://poc-vishing/v2/05_models/original/xgb_slow_learner/borderline_smote/10.pkl
  escrito s3://poc-vishing/v2/05_models/original/logistic_regression/borderline_smote/10.pkl
  escrito s3://poc-vishing/v2/05_models/original/random_forest/borderline_smote/10.pkl
  escrito s3://poc-vishing/v2/05_models/original/mlp/borderline_smote/10.pkl
  borderline_smote       20   (71406 filas)
  escrito s3://poc-vishing/v2/05_models/original/xgb_base/borderline_smote/20.pkl
  escrito s3:

,data_type,technique,ratio,variante,umbral,pr_auc,roc_auc,recall,precision,f1,recall_p90
0,original,none,0,xgb_shallow,0.8464,0.8504,0.9745,0.8218,0.8065,0.8141,0.6146
1,original,random_oversampling,10,xgb_shallow,0.8526,0.8497,0.9756,0.8058,0.8131,0.8095,0.6176
2,original,random_oversampling,25,xgb_shallow,0.8668,0.8493,0.9743,0.8098,0.8172,0.8135,0.6366
3,original,random_oversampling,20,xgb_shallow,0.8500,0.8486,0.9751,0.8088,0.8104,0.8096,0.6396
4,original,random_oversampling,25,xgb_regularized,0.8298,0.8473,0.9725,0.7988,0.8218,0.8102,0.6186
5,original,random_oversampling,25,xgb_conservative,0.7509,0.8466,0.9736,0.8058,0.8074,0.8066,0.6256
6,original,random_oversampling,20,xgb_regularized,0.8196,0.8459,0.9741,0.7998,0.8178,0.8087,0.6146
7,original,random_oversampling,25,xgb_deep,0.6380,0.8459,0.9726,0.7948,0.8135,0.8041,0.5906
8,original,none,0,xgb_regularized,0.7682,0.8457,0.9725,0.8178,0.8002,0.8089,0.5636
9,original,random_oversampling,10,xgb_regularized,0.7487,0.8455,0.9727,0.8168,0.8000,0.8083,0.6086


  escrito s3://poc-vishing/v2/06_results/validation_leaderboard.csv  (260 filas)


's3://poc-vishing/v2/06_results/validation_leaderboard.csv'

## Selección del ganador

Un único modelo pasa a R5. Se elige por PR-AUC sobre validation; si hay empate
técnico, desempata `recall@P=0.90`, que es la métrica operativa.

In [8]:
mejor = lb.iloc[0]
print("=== ganador sobre VALIDATION ===")
for k, v in mejor.items():
    print("  %-12s: %s" % (k, v))

print()
print("=== mejor por familia de algoritmo ===")
display(lb.loc[lb.groupby("variante").pr_auc.idxmax()]
          .sort_values("pr_auc", ascending=False))

print()
print("=== ¿sigue ganando random oversampling una vez cerrada la fuga? ===")
display(lb.groupby(["data_type", "technique"]).pr_auc.max().unstack(0).round(4))

=== ganador sobre VALIDATION ===
  data_type   : original
  technique   : none
  ratio       : 0
  variante    : xgb_shallow
  umbral      : 0.8464
  pr_auc      : 0.8504
  roc_auc     : 0.9745
  recall      : 0.8218
  precision   : 0.8065
  f1          : 0.8141
  recall_p90  : 0.6146

=== mejor por familia de algoritmo ===


,data_type,technique,ratio,variante,umbral,pr_auc,roc_auc,recall,precision,f1,recall_p90
0,original,none,0,xgb_shallow,0.8464,0.8504,0.9745,0.8218,0.8065,0.8141,0.6146
4,original,random_oversampling,25,xgb_regularized,0.8298,0.8473,0.9725,0.7988,0.8218,0.8102,0.6186
5,original,random_oversampling,25,xgb_conservative,0.7509,0.8466,0.9736,0.8058,0.8074,0.8066,0.6256
7,original,random_oversampling,25,xgb_deep,0.6380,0.8459,0.9726,0.7948,0.8135,0.8041,0.5906
14,original,random_oversampling,20,xgb_balanced,0.8453,0.8437,0.9718,0.7768,0.8407,0.8075,0.5876
17,original,random_oversampling,20,xgb_base,0.7500,0.8419,0.9728,0.8228,0.7859,0.8039,0.6156
25,original,none,0,logistic_regression,0.2869,0.8350,0.9703,0.7938,0.8100,0.8018,0.6046
30,original,random_oversampling,20,xgb_slow_learner,0.7554,0.8332,0.9732,0.8218,0.7760,0.7982,0.5616
70,original,random_oversampling,10,random_forest,0.2952,0.8220,0.9703,0.8158,0.7762,0.7955,0.5325
198,augmented,none,0,mlp,0.5734,0.7780,0.9424,0.7107,0.8041,0.7545,0.5045



=== ¿sigue ganando random oversampling una vez cerrada la fuga? ===


data_type,augmented,original
technique,,
borderline_smote,0.8087,0.8334
none,0.8277,0.8504
random_oversampling,0.8287,0.8497
smote,0.8031,0.8325
smote_undersampling,0.8059,0.8368


In [ ]:
vc.write_json({"ganador": {k: (v.item() if hasattr(v, "item") else v)
                           for k, v in mejor.items()}},
              vc.P.val_leaderboard.replace("validation_leaderboard.csv", "ganador.json"))
print("R4 completo. Continuar con R5 (única apertura del test).")

  escrito s3://poc-vishing/v2/06_results/ganador.json
R4 completo. Continuar con R5 (única apertura del test).
